PCA - метод, позволяющий уменьшить количество фичей в данных, не сильно теряя в информации, поскольку фичи могут сильно коррелировать между собой

In [14]:
import time
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

In [7]:
df = pd.read_csv('../datasets/YearPredictionMSD.txt')
df.head()

,2001,49.94357,21.47114,73.07750,8.74861,-17.40628,-13.09905,-25.01202,-12.23257,7.83089,...,13.01620,-54.40548,58.99367,15.37344,1.11144,-23.08793,68.40795,-1.82223,-27.46348,2.26327
0,2001,48.73215,18.42930,70.32679,12.94636,-10.32437,-24.83777,8.76630,-0.92019,18.76548,...,5.66812,-19.68073,33.04964,42.87836,-9.90378,-32.22788,70.49388,12.04941,58.43453,26.92061
1,2001,50.95714,31.85602,55.81851,13.41693,-6.57898,-18.54940,-3.27872,-2.35035,16.07017,...,3.03800,26.05866,-50.92779,10.93792,-0.07568,43.20130,-115.00698,-0.05859,39.67068,-0.66345
2,2001,48.24750,-1.89837,36.29772,2.58776,0.97170,-26.21683,5.05097,-10.34124,3.55005,...,34.57337,-171.70734,-16.96705,-46.67617,-12.51516,82.58061,-72.08993,9.90558,199.62971,18.85382
3,2001,50.97020,42.20998,67.09964,8.46791,-15.85279,-16.81409,-12.48207,-9.37636,12.63699,...,9.92661,-55.95724,64.92712,-17.72522,-1.49237,-7.50035,51.76631,7.88713,55.66926,28.74903
4,2001,50.54767,0.31568,92.35066,22.38696,-25.51870,-19.04928,20.67345,-5.19943,3.63566,...,6.59753,-50.69577,26.02574,18.94430,-0.33730,6.09352,35.18381,5.00283,-11.02257,0.02263


In [8]:
y = df.iloc[:, 0]
X = df.iloc[:, 1:]
#не использую train_test_split, поскольку доверился создателям датафрейма
X_train = X.iloc[:463715]
y_train = y.iloc[:463715]

X_test = X.iloc[463715:]
y_test = y.iloc[463715:]

тут я постараюсь реализовать собственный pca, который я сравню с реализацией в sklearn. Взял датасет https://archive.ics.uci.edu/dataset/203/yearpredictionmsd, чтобы можно было увидеть работу pca. 
на eda пофиг, я закину всё так, не буду даже пытаться анализировать фичи здесь

In [16]:
model = DecisionTreeRegressor()
start = time.perf_counter()
model.fit(X_train, y_train)
end = time.perf_counter()
print(f'{end-start} секунд время обучения')

75.32125789999998 секунд время обучения


In [17]:
from sklearn.metrics import root_mean_squared_error
y_pred = model.predict(X_test)
rmse = root_mean_squared_error(y_test, y_pred)
print(rmse)

13.423965009790898


In [18]:
from sklearn.linear_model import LinearRegression#вообще, можно использовать и Lasso, и Ridge, и их комбинацию
                                                    #но мне почему-то впадлу и хочется обычный линрег юзнуть
                                                    #линейные модели чувствительнее к размерности 
model = LinearRegression()
start = time.perf_counter()
model.fit(X_train, y_train)
end = time.perf_counter()
print(f'{end-start} секунд время обучения')

2.7918445999998767 секунд время обучения


In [20]:
from sklearn.metrics import root_mean_squared_error
y_pred = model.predict(X_test)
rmse = root_mean_squared_error(y_test, y_pred)
print(rmse)

9.510102360321458


In [21]:
from sklearn.linear_model import ElasticNet#А это линрег с l1- и l2 регуляризациями
model = ElasticNet()
start = time.perf_counter()
model.fit(X_train, y_train)
end = time.perf_counter()
print(f'{end-start} секунд время обучения')

6.5503960999999435 секунд время обучения


In [22]:
from sklearn.metrics import root_mean_squared_error
y_pred = model.predict(X_test)
rmse = root_mean_squared_error(y_test, y_pred)
print(rmse)

9.520920251112855


Сначала попробуем просто проскейлить дату

In [23]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [24]:
model = LinearRegression()
start = time.perf_counter()
model.fit(X_train, y_train)
end = time.perf_counter()
print(f'{end-start} секунд время обучения')

2.661732000000029 секунд время обучения


In [25]:
y_pred = model.predict(X_test)
rmse = root_mean_squared_error(y_test, y_pred)
print(rmse)

9.510102360321461


In [26]:
model = ElasticNet()
start = time.perf_counter()
model.fit(X_train, y_train)
end = time.perf_counter()
print(f'{end-start} секунд время обучения')

0.6178512999999839 секунд время обучения


In [27]:
y_pred = model.predict(X_test)
rmse = root_mean_squared_error(y_test, y_pred)
print(rmse)

10.211670959039884


In [28]:
model = DecisionTreeRegressor()
start = time.perf_counter()
model.fit(X_train, y_train)
end = time.perf_counter()
print(f'{end-start} секунд время обучения')

75.59384539999996 секунд время обучения


In [29]:
y_pred = model.predict(X_test)
rmse = root_mean_squared_error(y_test, y_pred)
print(rmse)

13.460772335671704


Странно, что ElasticNet так ответила на скейлер. Попробуем PCA в sklearn

In [30]:
pca = PCA(n_components=10)#первый случай - крайний, я оставляю только ~11% фичей
X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)

In [31]:
                                                    #линейные модели чувствительнее к размерности 
model = LinearRegression()
start = time.perf_counter()
model.fit(X_train, y_train)
end = time.perf_counter()
print(f'{end-start} секунд время обучения')
y_pred = model.predict(X_test)
rmse = root_mean_squared_error(y_test, y_pred)
print(rmse)#ответ сразу! но rmse скакнуло, что ожидаемо, ведь мы убрали ~89% фичей

0.124884800000018 секунд время обучения
10.496179948585324


In [32]:
model = ElasticNet()
start = time.perf_counter()
model.fit(X_train, y_train)
end = time.perf_counter()
print(f'{end-start} секунд время обучения')
y_pred = model.predict(X_test)
rmse = root_mean_squared_error(y_test, y_pred)
print(rmse)#аналогично

0.07069400000000314 секунд время обучения
10.55544689321817


In [33]:
model = DecisionTreeRegressor()
start = time.perf_counter()
model.fit(X_train, y_train)
end = time.perf_counter()
print(f'{end-start} секунд время обучения')
y_pred = model.predict(X_test)
rmse = root_mean_squared_error(y_test, y_pred)
print(rmse)#ответ тоже есть! Картина аналогична линейным моделям

9.19091279999975 секунд время обучения
14.63015276560606


In [34]:
X_train = X.iloc[:463715]
y_train = y.iloc[:463715]

X_test = X.iloc[463715:]
y_test = y.iloc[463715:]

pca = PCA(n_components=45)#второй случай - где-то посередине, оставляю 50% фичей
X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)

In [35]:
model = LinearRegression()
start = time.perf_counter()
model.fit(X_train, y_train)
end = time.perf_counter()
print(f'{end-start} секунд время обучения')
y_pred = model.predict(X_test)
rmse = root_mean_squared_error(y_test, y_pred)
print(rmse)

0.8836759999999231 секунд время обучения
10.362827799954198


In [36]:
model = ElasticNet()
start = time.perf_counter()
model.fit(X_train, y_train)
end = time.perf_counter()
print(f'{end-start} секунд время обучения')
y_pred = model.predict(X_test)
rmse = root_mean_squared_error(y_test, y_pred)
print(rmse)

0.2993854999999712 секунд время обучения
10.36261570095756


In [37]:
model = DecisionTreeRegressor()
start = time.perf_counter()
model.fit(X_train, y_train)
end = time.perf_counter()
print(f'{end-start} секунд время обучения')
y_pred = model.predict(X_test)
rmse = root_mean_squared_error(y_test, y_pred)
print(rmse)

37.94742240000005 секунд время обучения
14.89757835036871


думаю очев, что чем больше фич мы оставляем, тем меньше выигрыша по времени от pca, но тем меньше ухудшается метрика. Теперь реализуем pca самостоятельно(а ещё я забыл отскейлить во втором случае дату, но это так)

In [59]:
def pca_fit(X, n_components):
    X_mean = np.mean(X, axis=0)
    X_centered = X - X_mean
    n = X.shape[0]
    U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)
    components = Vt[:n_components]
    S_sq = S ** 2
    explained_variance_ratio = S_sq[:n_components] / np.sum(S_sq)
    projected = U[::n_components] * S[n_components]
    return X_mean, components, explained_variance_ratio
def pca_transform(X, X_mean, components):
    X_centered = X - X_mean
    return X_centered @ components.T

### Что делаем после SVD?

После сингулярного разложения

$$
X_{centered} = U\Sigma V^T
$$

сингулярные числа $\sigma_i$ связаны с собственными значениями
ковариационной матрицы:

$$
\lambda_i = \frac{\sigma_i^2}{n-1}
$$

Сингулярные числа упорядочены по убыванию, поэтому первые столбцы
$U$ соответствуют направлениям с наибольшей дисперсией.

Для PCA оставляем только первые $k$ компонент:

$$
U_k = U[:, :k], \qquad \Sigma_k = \Sigma[:k]
$$

и получаем новые признаки:

$$
X_{PCA} = U_k\Sigma_k
$$

Таким образом, вместо исходных $d$ признаков получаем $k$ новых
ортогональных признаков, сохраняя направления с наибольшей дисперсией.

Доля сохранённой дисперсии:

$$
\text{Explained Variance Ratio}
=
\frac{\sum_{i=1}^{k}\lambda_i}
{\sum_{i=1}^{d}\lambda_i}
$$

Чем меньше $k$, тем сильнее уменьшается размерность и тем больше
информации мы потенциально теряем.

In [60]:
X_train = X.iloc[:463715]
y_train = y.iloc[:463715]

X_test = X.iloc[463715:]
y_test = y.iloc[463715:]

mean, components, evr = pca_fit(X_train, 10)

X_train_pca = pca_transform(
    X_train, mean, components
)

X_test_pca = pca_transform(
    X_test, mean, components
)

(малость долговато было, но тем не менее. Таймер смотреть не хочу)

In [61]:
model = ElasticNet()
start = time.perf_counter()
model.fit(X_train_pca, y_train)
end = time.perf_counter()
print(f'{end-start} секунд время обучения')
y_pred = model.predict(X_test_pca)
rmse = root_mean_squared_error(y_test, y_pred)
print(rmse)

0.07409810000035577 секунд время обучения
10.659670837136076


(по хорошему бы сделать табличку со всеми резами, но мне наверное слишком лень)

## Выводы

- PCA позволяет уменьшить размерность данных.
- Чем меньше компонент оставляем, тем быстрее обучение, но тем сильнее может ухудшаться качество.
- Линейные модели оказались чувствительнее к уменьшению размерности.
- PCA в sklearn и собственная реализация через SVD дают сопоставимый результат.